<a href="https://colab.research.google.com/github/yaswithapodamekala/infosys-springboard-Intern/blob/main/infosys.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install streamlit pyngrok bcrypt pyjwt transformers evaluate --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 28.0 MB/s eta 0:00:00


In [2]:
!ngrok config add-authtoken 39brCnJgpbYyKpVjyea4CFgTh3r_5QMrebYhEgYbpSEWbj9Fw


Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [3]:
%%writefile app.py
import streamlit as st
import sqlite3
import bcrypt
import jwt
import datetime
SECRET_KEY = "textmorph_super_secret_key_123456"
# ================= DATABASE =================

def connect_db():
    return sqlite3.connect("users.db")

conn = connect_db()
c = conn.cursor()
c.execute("""
CREATE TABLE IF NOT EXISTS users (
    email TEXT PRIMARY KEY,
    password BLOB,
    question TEXT,
    answer TEXT
)
""")
conn.commit()
conn.close()

def add_user(email, password, question, answer):
    conn = connect_db()
    c = conn.cursor()
    c.execute("INSERT INTO users (email, password, question, answer) VALUES (?,?,?,?)",
              (email, password, question, answer))
    conn.commit()
    conn.close()

def get_user(email):
    conn = connect_db()
    c = conn.cursor()
    c.execute("SELECT * FROM users WHERE email=?", (email,))
    user = c.fetchone()
    conn.close()
    return user

def update_password(email, new_pass):
    conn = connect_db()
    c = conn.cursor()
    c.execute("UPDATE users SET password=? WHERE email=?",
              (new_pass, email))
    conn.commit()
    conn.close()

# ================= JWT =================

def create_token(email):
    payload = {
        "user": email,
        "exp": datetime.datetime.utcnow() + datetime.timedelta(hours=2)
    }
    return jwt.encode(payload, SECRET_KEY, algorithm="HS256")

def verify_token(token):
    try:
        decoded = jwt.decode(token, SECRET_KEY, algorithms=["HS256"])
        return decoded["user"]
    except:
        return None

# ================= UI =================

st.set_page_config(page_title="TextMorph System")

menu = ["Signup", "Login", "Forgot Password"]
choice = st.sidebar.selectbox("Menu", menu)

# ================= SIGNUP =================

if choice == "Signup":

    st.title("Infosys Springboard Intern - Signup")

    email = st.text_input("Email")
    password = st.text_input("Password", type="password")
    confirm = st.text_input("Confirm Password", type="password")

    question = st.selectbox("Security Question", [
        "What is your pet name?",
        "What is your favorite teacher?",
        "What is your mother's name?"
    ])

    answer = st.text_input("Answer")

    if st.button("Signup"):

        if email == "" or password == "" or answer == "":
            st.error("All fields are mandatory!")

        elif password != confirm:
            st.error("Passwords not matching!")

        elif get_user(email):
            st.warning("User already exists!")

        else:
            hashed = bcrypt.hashpw(password.encode(), bcrypt.gensalt())
            add_user(email, hashed, question, answer)
            st.success("Signup Successful")

# ================= LOGIN =================

elif choice == "Login":

    st.title("Login Page")

    email = st.text_input("Email")
    password = st.text_input("Password", type="password")

    if st.button("Login"):

        user = get_user(email)

        if user is None:
            st.error("User not found!")

        else:
            db_pass = user[1]

            if bcrypt.checkpw(password.encode(), db_pass):

                token = create_token(email)

                st.session_state["user"] = email
                st.session_state["token"] = token

                st.success("Login Successful")
                st.code(token)

            else:
                st.error("Wrong password")


# ================= FORGOT PASSWORD =================

elif choice == "Forgot Password":

    st.title("Reset Password")

    email = st.text_input("Enter Email")

    if st.button("Verify Email"):

        user = get_user(email)

        if user is None:
            st.error("Email not found!")

        else:
            st.session_state["reset_email"] = email
            st.session_state["question"] = user[2]
            st.success("Email verified")

    if "reset_email" in st.session_state:

        st.info(st.session_state["question"])

        ans = st.text_input("Your Answer")
        new_pass = st.text_input("New Password", type="password")

        if st.button("Reset Password"):

            user = get_user(st.session_state["reset_email"])

            if ans == user[3]:

                hashed = bcrypt.hashpw(new_pass.encode(), bcrypt.gensalt())
                update_password(st.session_state["reset_email"], hashed)

                st.success("Password Updated")
                del st.session_state["reset_email"]

            else:
                st.error("Wrong answer")


Writing app.py


In [6]:
!streamlit run app.py --server.port 8501 &





  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.75.93.250:8501

  Stopping...


In [5]:
from pyngrok import ngrok

ngrok.kill()  # clear old tunnels
public_url = ngrok.connect(8501)
print("🚀 Open this URL:", public_url)


🚀 Open this URL: NgrokTunnel: "https://untwilled-barbera-presurgery.ngrok-free.dev" -> "http://localhost:8501"
